In [5]:
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd
import os
import sys

sys.path.append("..")

from settings import (
    random_state,
    PROJECT_PATH,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)
from sklearn.ensemble import RandomForestRegressor
import shap
from shap import TreeExplainer


In [14]:
transactions = pl.read_parquet(
    os.path.join(PROJECT_PATH, "real_estate_transactions_engineered.parquet")
)

selected_region = "region_occitanie"
region_transactions = transactions.filter(pl.col(selected_region) == 1)

X = region_transactions.drop([REGRESSION_TARGET, CLASSIFICATION_TARGET]).to_pandas()
y_regression = region_transactions[REGRESSION_TARGET].to_pandas()

In [15]:
feature_names_simplified = [
    "living_area",
    "avg_price_per_m2_previous_month",
    "longitude",
    "latitude",
    "num_transactions_previous_month",
    "building_type_apartment",
]

In [16]:
rf_regressor_light = RandomForestRegressor(random_state=random_state)
rf_regressor_light.fit(X[feature_names_simplified], y_regression)

RandomForestRegressor(random_state=42)

In [23]:
explainer = TreeExplainer(rf_regressor_light, approximate=True)
shap_values = explainer(X[feature_names_simplified])

This is the reference value that SHAP uses to compute Shapley values 

In [27]:

explainer.expected_value

array([127597.1438447])

In [20]:
y_pred = rf_regressor_light.predict(X[feature_names_simplified])

absolute_errors_train = pd.Series(
    [
        abs(true_value - predicted_value)
        for (true_value, predicted_value) in zip(y_regression, y_pred)
    ]
)

Here, we see that the average model error is around €11k. Shapley value interpretations of this magnitude should be taken with caution, as it becomes difficult to distinguish a genuine feature effect from estimation error. 

In [21]:
print(absolute_errors_train.describe().apply(lambda x: format(x, "f")))

count     50982.000000
mean      10924.996767
std       16886.158663
min           0.000000
25%        2682.100000
50%        6333.200000
75%       13237.383750
max      774482.200000
dtype: object


The Dependency Plot below allows us to interpret the relationship between the variable "living_area" and the Shapley values, while also correlating with a third variable, in order to detect more complex behavioural patterns than those visible in the Beeswarm chart!

The chart shows the dispersion of Shapley values for different values of "living_area". We can observe that the Shapley values for the "living_area" variable are predominantly positive, indicating a positive influence of this feature on the model's predictions. This relationship also appears linear, meaning that higher values of "living_area" are generally associated with higher predicted prices. This is a perfectly coherent interpretation from a real-estate business perspective.

Within this linear trend, no distinct behavioural clusters emerge with respect to the second feature "avg_price_per_m2_previous_month". The correlation in question is therefore purely numerical and does not reflect any particular business reality.

However, there are also a few observations with negative Shapley values, suggesting a negative influence of the "living_area" feature on the model's predictions. This may be due to outliers, or to the effect of other features in the model that are not visible here.

This is indeed the limitation of this type of chart: we visualise the Shapley values and 2 features while assuming all other features are held constant — a very strong assumption, given that correlations with other features should not be ignored.


In [ ]:
shap.plots.scatter(shap_values[:, "living_area"], color=shap_values)

In this chart, by contrast, we can draw almost no conclusions:
* Regardless of the values taken by the "num_transactions_previous_month" feature, the amplitude of the Shapley values remains the same, broadly ranging between -€50k and €50k.
* The second feature also does not help explain why the contribution of the first feature is positive in some cases and negative in others.
* The effects observed are most likely due to other features that are not visualised here.


In [ ]:

shap.plots.scatter(shap_values[:, "num_transactions_previous_month"], color=shap_values)